In [1]:
import torch
import pandas as pd
import numpy as np
from transformers import BartTokenizer, BartForSequenceClassification

MODEL_DIR = "./bart_relevancia_model"
MAX_LENGTH = 256

tokenizer = BartTokenizer.from_pretrained(MODEL_DIR)
model = BartForSequenceClassification.from_pretrained(MODEL_DIR)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

def classificar_textos(textos, limiar_relevante=0.85, limiar_revisao=0.40):
    inputs = tokenizer(
        textos,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=MAX_LENGTH,
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()

    resultados = []
    for texto, prob in zip(textos, probs):
        prob_nao_relevante = float(prob[0])
        prob_relevante = float(prob[1])

        if prob_relevante >= limiar_relevante:
            classificacao = 1
            status = "relevante"
        elif prob_relevante >= limiar_revisao:
            classificacao = ""
            status = "revisar"
        else:
            classificacao = 0
            status = "nao_relevante"

        resultados.append({
            "texto": texto,
            "classificacao_final": classificacao,
            "status": status,
            "prob_nao_relevante": round(prob_nao_relevante, 6),
            "prob_relevante": round(prob_relevante, 6),
        })

    return pd.DataFrame(resultados)

# Exemplo de uso
textos = [
    "Estação Solda tipo corrente: alternada, tensão alimentação: 110, tipo ponta: removível",
    "Locação de catraca eletrônica tipo pedestal de 03 braços com software gerenciador",
    "Aquisição de medicamentos hospitalares",
    "Terminal de controle de acesso multibiométrico com reconhecimento facial",
]

df_resultado = classificar_textos(textos)
print(df_resultado)

You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.


Loading weights:   0%|          | 0/263 [00:00<?, ?it/s]

                                               texto classificacao_final  \
0  Estação Solda tipo corrente: alternada, tensão...                       
1  Locação de catraca eletrônica tipo pedestal de...                   1   
2             Aquisição de medicamentos hospitalares                   0   
3  Terminal de controle de acesso multibiométrico...                   1   

          status  prob_nao_relevante  prob_relevante  
0        revisar            0.190402        0.809598  
1      relevante            0.000011        0.999989  
2  nao_relevante            0.999888        0.000112  
3      relevante            0.002123        0.997877  


Se você tiver um arquivo com uma coluna texto:

In [ ]:
import pandas as pd

df = pd.read_csv("entradas.csv")

resultados = classificar_textos(df["texto"].astype(str).tolist())

resultados.to_csv("saida_classificada.csv", index=False, encoding="utf-8-sig")
print("Arquivo salvo: saida_classificada.csv")